In [2]:
import pandas as pd
import numpy as np

def process_abundance(abundance_path: str, cluster_path: str):
    """
    Process a large metagenomic abundance TSV file to produce normalized abundances aggregated by cluster.

    Parameters:
        abundance_path (str): Path to the contig abundance TSV file. 
                               The first column is contig ID (e.g., "S<sample_name>C<contig_name>"), 
                               subsequent columns are per-sample abundance values.
        cluster_path (str): Path to the clustering results TSV file (two columns: cluster_label and contig).

    Returns:
        np.ndarray: 2D array of shape (n_clusters, n_samples) with normalized abundance sums per cluster.
                    Row i corresponds to cluster i, columns correspond to samples.
    """
    # STACKOVERFLOW CHUNCKSIZE READER:
    # chunksize = 10 ** 6
    # with pd.read_csv(filename, chunksize=chunksize) as reader:
    #     for chunk in reader:
    #         process(chunk)


    ### 1. Compute sample_depths_sum (total abundance per sample) in a first pass ###
    sample_depths_sum = None
    chunk_iter = pd.read_csv(abundance_path, sep='\t', index_col=0, chunksize=10000)
    
    for chunk in chunk_iter: # iterrows
        # Ensure abundance values are numeric (float). Use float64 for precision.
        data = chunk.astype(np.float64) #32 for less memory
        
        col_sums = data.sum(axis=0)  # sum each sample column in this chunk
        if sample_depths_sum is None:
            sample_depths_sum = col_sums
        else:
            sample_depths_sum += col_sums

    # Convert to numpy array for faster operations and check for zeros
    sample_depths_sum = sample_depths_sum.values  # shape (n_samples,)
    
    if (sample_depths_sum == 0).any():
        raise ValueError("One or more samples have zero total abundance, cannot normalize by depth.")
    
    
    n_samples = len(sample_depths_sum)

    #################################################
    ### 2. Load cluster mapping into a dictionary ###
    #################################################


    cluster_df = pd.read_csv(cluster_path, sep='\t', header=None, names=['cluster', 'contig'])
   
   
    # Create mapping from contig -> cluster index
    unique_clusters = pd.unique(cluster_df['cluster'])

    cluster_to_idx = {cluster: idx for idx, cluster in enumerate(unique_clusters)}

    contig_to_cluster_idx = {row['contig']: cluster_to_idx[row['cluster']] 
                              for _, row in cluster_df.iterrows()}

    n_clusters = len(unique_clusters)

    # Initialize result array for aggregated cluster abundances
    result = np.zeros((n_clusters, n_samples), dtype=np.float32)  # using float32 to save memory; use float64 if needed

    ############################################################################
    ### 3. Second pass: process chunks to normalize and aggregate by cluster ###
    ############################################################################
    
    
    chunk_iter = pd.read_csv(abundance_path, sep='\t', index_col=0, chunksize=10000)
    for chunk in chunk_iter:
        data = chunk.astype(np.float32).values  # convert chunk to NumPy array of shape (rows, n_samples)

        # (a) Normalize by sample depth (counts per million scaling)
        data *= (1000000.0 / sample_depths_sum)  # broadcast division by each sample's depth&#8203;:contentReference[oaicite:11]{index=11}
        
        # (b) Normalize each contig's distribution to sum to 1
        row_sums = data.sum(axis=1) # total abundance per contig (after depth normalization)
        
        zero_rows = (row_sums == 0)
        if np.any(zero_rows):
            # Set zero-sum contig rows to 1/n_samples each&#8203;:contentReference[oaicite:12]{index=12}
            data[zero_rows, :] = 1.0 / n_samples
            row_sums[zero_rows] = 1.0
        # Divide by row sums to get proportion per contig&#8203;:contentReference[oaicite:13]{index=13}
        data /= row_sums[:, np.newaxis]          # each contig row now sums to ~1
        # (c) Aggregate contigs into their clusters
        contig_names = chunk.index.values
        # Use a loop (or vectorized add) to accumulate each contig's contributions to its cluster
        for i, contig in enumerate(contig_names):
            cluster_idx = contig_to_cluster_idx.get(contig)
            if cluster_idx is not None:
                result[cluster_idx] += data[i]
        # (Optionally, for speed, one could vectorize the above loop with np.add.at)
    # End of chunk processing loop

    ### 4. Return the aggregated result ###
    # The result is a NumPy array with shape (n_clusters, n_samples).
    # If a PyTorch tensor is needed, you can do: torch.tensor(result)
    return result
